In [ ]:
import os
import base64
import csv
import fitz  
import chromadb
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

True

In [47]:
client = OpenAI(
    base_url="https://api.swissai.svc.cscs.ch/v1",
    api_key=os.getenv("CSCS_API_KEY"),
)

EMBED_MODEL  = "Snowflake/snowflake-arctic-embed-l-v2.0"
VISION_MODEL = "moonshotai/Kimi-K2.5-SDSC"
TEXT_MODEL = "swiss-ai/Apertus-8B-Instruct-2509"

PDF_FILES = [
    "a2q-answer-any-question-dataset/natural-catastrophe-and-climate-report-2023.pdf",
    "a2q-answer-any-question-dataset/swissre_sigma-1_2024_english.pdf",
    "a2q-answer-any-question-dataset/Web Version _E-Government Survey 2024 11102024.pdf",
    "a2q-answer-any-question-dataset/World_Inequality_Report_2026.pdf",
]
QA_CSV     = "Durham-Hackathon-2026-template/data/a2q/qa_set.csv"
OUTPUT_CSV = "answers_output.csv"
CHROMA_DIR = "./chroma_db"


In [3]:
def embed_text(text: str) -> list:
    try:
        resp = client.embeddings.create(model=EMBED_MODEL, input=text[:8000])
        return resp.data[0].embedding
    except Exception as e:
        print(f"  [embed error] {e}")
        return [0.0] * 1024

In [12]:
def page_to_base64(page) -> str:
    pix = page.get_pixmap(dpi=150)
    png_bytes = pix.tobytes("png")
    b64 = base64.b64encode(png_bytes).decode("utf-8")
    return f"data:image/png;base64,{b64}"


def describe_page_image(page, page_num: int, doc_name: str) -> str:
    try:
        img_uri = page_to_base64(page)
        resp = client.chat.completions.create(
            model=VISION_MODEL,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": (
                        f"This is page {page_num} from '{doc_name}'. "
                        "Describe ALL text, tables, charts, and figures in detail, "
                        "including every number, label, axis value, legend, and trend. "
                        "Be extremely thorough so the content can be used for question answering."
                    )},
                    {"type": "image_url", "image_url": {"url": img_uri}},
                ],
            }],
            max_tokens=1500,
            temperature=0.0,
        )
        return resp.choices[0].message.content
    except Exception as e:
        return f"[vision error on page {page_num}: {e}]"

In [ ]:
def has_images(page) -> bool:
    img_list = page.get_images(full=True)
    if not img_list:
        return False
    
    page_area = page.rect.width * page.rect.height
    total_img_area = 0
    
    for img in img_list:
        if img[2] * img[3] < 90000: 
            continue
        return True
        
    return False

In [5]:
def extract_pdf_chunks(pdf_path: str, chunk_size: int = 400, overlap: int = 100) -> list:
    doc_name = os.path.basename(pdf_path)
    chunks = []
    doc = fitz.open(pdf_path)
    print(f"  Processing {doc_name} ({len(doc)} pages)...")

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text").strip()

        if has_images(page):
            print(f"    Page {page_num}: has images, calling vision model...")
            vision_desc = describe_page_image(page, page_num, doc_name)
            combined = f"[Page {page_num} text]\n{text}\n\n[Page {page_num} visual content]\n{vision_desc}"
        else:
            combined = f"[Page {page_num}]\n{text}"

        words = combined.split()
        for i in range(0, len(words), chunk_size - overlap):
            chunk_words = words[i: i + chunk_size]
            chunk_text = " ".join(chunk_words).strip()
            if len(chunk_text) > 50:
                chunks.append({
                    "text": chunk_text,
                    "source": doc_name,
                    "page": page_num,
                    "id": f"{doc_name}_p{page_num}_c{i}",
                })

    doc.close()
    print(f"    -> {len(chunks)} chunks extracted")
    return chunks

In [6]:
def build_index(chunks: list, collection) -> None:
    batch_size = 50
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i: i + batch_size]
        texts      = [c["text"]   for c in batch]
        ids        = [c["id"]     for c in batch]
        metas      = [{"source": c["source"], "page": c["page"]} for c in batch]
        embeddings = [embed_text(t) for t in texts]
        collection.upsert(ids=ids, documents=texts, embeddings=embeddings, metadatas=metas)
        print(f"  Indexed chunks {i}-{i+len(batch)-1}")

In [61]:
def retrieve(question: str, collection, top_k: int = 5) -> list:
    q_emb = embed_text(question)
    results = collection.query(query_embeddings=[q_emb], n_results=top_k)
    return results["documents"][0]

In [ ]:
def answer_question(question: str, context_chunks: list) -> str:
    valid_chunks = [c.strip() for c in context_chunks if c][:5]
    context = "\n\n---\n\n".join(valid_chunks)
    
    prompt = (
        "Answer the question using the provided context strictly. Be short (1-2 sentences).\n\n"
        f"CONTEXT:\n{context}\n\n"
        f"QUESTION: {question}\n\n"
        "ANSWER:"
    )
    try:
        resp = client.chat.completions.create(
            model=TEXT_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=150,
            temperature=0.0,
        )
        
        content = resp.choices[0].message.content
        if not content or len(content.strip()) == 0:
            return "API returned empty. Try switching TEXT_MODEL to swiss-ai/Apertus-8B-Instruct-2509"
            
        return content.strip()
    except Exception as e:
        return f"[Error] {e}"

In [9]:
def load_questions(csv_path: str) -> list:
    questions = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        sample = f.read(2048)
        f.seek(0)
        delimiter = ";" if sample.count(";") > sample.count(",") else ","
        reader = csv.DictReader(f, delimiter=delimiter)
        for row in reader:
            keys = list(row.keys())
            if len(keys) == 1:
                parts = list(row.values())[0].split(";", 1)
                q = parts[0].strip()
            else:
                q_col = next((k for k in keys if "question" in k.lower()), keys[0])
                q = row[q_col].strip()
            if q and q.lower() != "question":
                questions.append({"question": q})
    return questions

In [10]:
print("=== Setting up ChromaDB ===")
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = chroma_client.get_or_create_collection(
    name="hackathon_docs",
    metadata={"hnsw:space": "cosine"},
)

=== Setting up ChromaDB ===


In [18]:
existing_count = collection.count()
if existing_count == 0:
    print("=== Extracting and indexing PDFs ===")
    all_chunks = []
    for pdf_path in PDF_FILES:
        if not os.path.exists(pdf_path):
            print(f"  WARNING: {pdf_path} not found, skipping.")
            continue
        chunks = extract_pdf_chunks(pdf_path)
        all_chunks.extend(chunks)

    print(f"\nTotal chunks: {len(all_chunks)}")
    print("=== Building vector index ===")
    build_index(all_chunks, collection)
    print(f"Index built: {collection.count()} chunks stored.")
else:
    print(f"=== Using existing index ({existing_count} chunks) ===")

=== Extracting and indexing PDFs ===
  Processing natural-catastrophe-and-climate-report-2023.pdf (76 pages)...
    Page 1: has images, calling vision model...
    Page 3: has images, calling vision model...
    Page 5: has images, calling vision model...
    Page 7: has images, calling vision model...
    Page 12: has images, calling vision model...
    Page 18: has images, calling vision model...
    Page 28: has images, calling vision model...
    Page 39: has images, calling vision model...
    Page 45: has images, calling vision model...
    Page 46: has images, calling vision model...
    Page 47: has images, calling vision model...
    Page 58: has images, calling vision model...
    -> 138 chunks extracted
  Processing swissre_sigma-1_2024_english.pdf (37 pages)...
    -> 69 chunks extracted
  Processing Web Version _E-Government Survey 2024 11102024.pdf (206 pages)...
    Page 1: has images, calling vision model...
    Page 7: has images, calling vision model...
    Page 38: h

In [19]:
print(f"\n=== Loading questions from {QA_CSV} ===")
questions = load_questions(QA_CSV)
print(f"Loaded {len(questions)} questions.")


=== Loading questions from Durham-Hackathon-2026-template/data/a2q/qa_set.csv ===
Loaded 40 questions.


In [63]:
print("\n=== Answering questions ===")

results = []

for i, item in enumerate(questions, start=1):

    q = item["question"]

    print(f"Q{i}: {q[:80]}...")

    chunks = retrieve(q, collection, top_k=12)

    answer = answer_question(q, chunks)

    print(f"  A: {answer[:120]}...")

    results.append({"question": q, "answer": answer})




=== Answering questions ===
Q1: What year where there most casualties from man-made disasters in the recorded da...
  A: The year with the most casualties from man-made disasters in the recorded data is not explicitly stated in the provided ...
Q2: In what year did the quantity of man-made disasters peak in the recorded data be...
  A: The quantity of man-made disasters peaked in 2006, with 98 events recorded that year....
Q3: Between 1970 and 2023 what year in the data shows the largest number of natural ...
  A: The year with the largest number of natural catastrophes between 1970 and 2023 is 2023, with 142 loss-inducing natural d...
Q4: What year between 1994 and 2023 had the most high severity ($5 billion in damage...
  A: The year 2023 had the most high severity ($5 billion in damages or more) natural catastrophes....
Q5: How much higher are the 2023 insured losses than the previous 10 year average?...
  A: The 2023 insured losses are 11% higher than the previous 10-year average.

In [ ]:
NEW_OUTPUT_CSV = "answers_v4.csv" 

with open(NEW_OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["question", "answer"], delimiter=";")
    writer.writeheader()
    writer.writerows(results)

print(f"\n=== Done! : {NEW_OUTPUT_CSV} ===")

In [65]:
import pandas as pd

df = pd.read_csv("answers_v4.csv", encoding="utf-8", sep=";").fillna("")
df.head(10)

,question,answer
0,What year where there most casualties from man...,The year with the most casualties from man-mad...
1,In what year did the quantity of man-made disa...,The quantity of man-made disasters peaked in 2...
2,Between 1970 and 2023 what year in the data sh...,The year with the largest number of natural ca...
3,What year between 1994 and 2023 had the most h...,The year 2023 had the most high severity ($5 b...
4,How much higher are the 2023 insured losses th...,The 2023 insured losses are 11% higher than th...
5,Which figure shows the trend in insured losses...,The figure that shows the trend in insured los...
6,"Before 2023, what was the highest year on reco...","Before 2023, the highest year on record for Eu..."
7,What regions does the Swiss Re report on natur...,The Swiss Re report on natural catastrophes sp...
8,What is the highest Benefit to Cost ratio buil...,The highest Benefit to Cost ratio building cod...
9,According to the Swiss Re Institute report on ...,"According to the Swiss Re Institute report, th..."


In [ ]:
import pandas as pd
import requests

In [ ]:
TEAM_NAME = "AlphubeL"

In [ ]:
BASE_URL = "http://durham-leaderboard-runai-innovation-klemen.inference.compute.datascience.ch/"
leaderboard_endpoint = f"{BASE_URL}/api/v1/leaderboard"
submit_endpoint = f"{BASE_URL}/api/v1/submit"


def get_leaderboard() -> pd.DataFrame:
    leaderboard = requests.get(leaderboard_endpoint).json()
    return pd.DataFrame(leaderboard["entries"])


def submit(df: pd.DataFrame, user: str, token: str) -> pd.Series:
    if user == "your_team_name" or user == "":
        raise ValueError("Please set your team name in the 'TEAM_NAME' variable.")
    predictions = df[["question", "answer"]].to_dict(orient="records")
    submission = {"predictions": predictions}
    response = requests.post(submit_endpoint, json=submission, auth=(user, token))
    if (response.status_code // 100) != 2:
        response.raise_for_status()
    print(response.json())
    return pd.Series(response.json())

In [ ]:
print("BASE_URL:", BASE_URL)
print("leaderboard_endpoint:", leaderboard_endpoint)
print("submit_endpoint:", submit_endpoint)

r = requests.get(BASE_URL)
print("BASE_URL status:", r.status_code)
print(r.text[:500])

r = requests.get(leaderboard_endpoint)
print("leaderboard status:", r.status_code)
print(r.text[:500])

In [ ]:
import requests

BASE_URL = "http://durham-leaderboard-runai-innovation-klemen.inference.compute.datascience.ch"

paths = [
    "/leaderboard",
    "/submit",
    "/api/leaderboard",
    "/api/submit",
    "/api/v1/leaderboard",
    "/api/v1/submit",
    "/api/v1/qa/leaderboard",
    "/api/v1/qa/submit",
    "/docs",
    "/openapi.json"
]

for path in paths:
    url = BASE_URL + path
    try:
        r = requests.get(url)
        print(path, r.status_code, r.text[:120].replace("\n", " "))
    except Exception as e:
        print(path, "ERROR", e)

In [ ]:
import pandas as pd
import requests

TEAM_NAME = "AlphubeL"

BASE_URL = "http://durham-leaderboard-runai-innovation-klemen.inference.compute.datascience.ch"

leaderboard_endpoint = f"{BASE_URL}/api/v1/leaderboard"
submit_endpoint = f"{BASE_URL}/api/v1/submit"


def get_leaderboard() -> pd.DataFrame:
    response = requests.get(leaderboard_endpoint)
    data = response.json()
    print(data)
    return pd.DataFrame(data["entries"])


def submit(df: pd.DataFrame, user: str, token: str) -> pd.Series:
    if user == "your_team_name" or user == "":
        raise ValueError("Please set your team name in the 'TEAM_NAME' variable.")
    
    predictions = df[["question", "answer"]].to_dict(orient="records")
    submission = {"predictions": predictions}
    
    response = requests.post(
        submit_endpoint,
        json=submission,
        auth=(user, token)
    )
    
    if (response.status_code // 100) != 2:
        print(response.status_code)
        print(response.text)
        response.raise_for_status()
    
    print(response.json())
    return pd.Series(response.json())

In [ ]:
df = pd.read_csv("answers_v4.csv", encoding="utf-8", sep=";").fillna("")
df.head(10)

In [ ]:
submit(df, user=TEAM_NAME, token="cant_be_empty")